## Import libraries

In [1]:
import pandas as pd
import os
from obspy import read, Stream, UTCDateTime as utc
from glob import glob
from tqdm import tqdm

## Load HYPOINVERSE catalog

In [2]:
hypinv_table=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/kim1983/Imsil.sum", sep=',')
hypinv_table[["   DATE     TIME", " SEC ", "   LAT  ", "   LON   ", " DEPTH", "NUM", "GAP", " DMIN", " RMS ", "  ERH", "  ERZ"]].head()

,DATE TIME,SEC,LAT,LON,DEPTH,NUM,GAP,DMIN,RMS,ERH,ERZ
0,2025/11/02 03:06,7.33,35.5585,127.3033,21.13,9,165,6.2,0.78,3.5,5.5
1,2025/11/02 04:13,27.01,35.2357,127.3810,7.87,14,208,14.2,0.15,0.7,2.0
2,2025/11/02 06:20,48.35,35.4551,127.3748,1.25,8,113,26.0,0.92,2.4,65.3
3,2025/11/02 07:36,46.77,35.5937,127.2757,27.18,8,120,2.3,0.33,1.6,2.8
4,2025/11/02 10:42,33.87,35.5199,127.0868,0.63,7,147,14.6,0.31,0.9,24.8


## Load station table

In [11]:
used_station_list=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/station_table/used_stations_100km.csv", sep=",")
used_station_list.head()

,Network,Code,Latitude,Longitude,Elevation,Sensor
0,KS,BAU,35.1027,127.5968,562.0,EL
1,KS,BLGA,36.1392,127.3032,105.0,HG
2,KS,BLLA,34.8646,127.4638,45.0,HG
3,KS,BUYB,36.2726,126.9206,59.0,HH
4,KS,CGPA,34.8768,126.9710,144.0,HG


## Load continuous waveform

In [3]:
all_streams=Stream()

mseed_list=glob("../3.PhaseNO/waveforms/251102/KS.*.mseed")
for mseed_file in mseed_list:
    st=read(mseed_file)
    all_streams += st

In [4]:
all_streams.merge(method=1, fill_value=0)
print(all_streams.select(station="SIDA"))

3 Trace(s) in Stream:
KS.SIDA..HGE | 2025-11-02T00:00:00.000000Z - 2025-11-02T23:59:59.990000Z | 100.0 Hz, 8640000 samples
KS.SIDA..HGN | 2025-11-02T00:00:00.000000Z - 2025-11-02T23:59:59.990000Z | 100.0 Hz, 8640000 samples
KS.SIDA..HGZ | 2025-11-02T00:00:00.000000Z - 2025-11-02T23:59:59.990000Z | 100.0 Hz, 8640000 samples


## Trim the event waveforms and save as `SAC` files

In [5]:
t_before, t_after=30., 90.
output_dir="/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/event_waveforms"
os.makedirs(output_dir, exist_ok=True)

for i in tqdm(range(len(hypinv_table))):
    # Get origin times
    datetime=hypinv_table["   DATE     TIME"][i]
    second=hypinv_table[" SEC "][i]    
    year, month, day=int(datetime[:4]), int(datetime[5:7]), int(datetime[8:10])
    hour, minute, second, microsecond=int(datetime[11:13]), int(datetime[14:16]), int(second), int((second-int(second))*1e6)
    origin_time=utc(year, month, day, hour, minute, second, microsecond)
    
    t_start=origin_time-t_before
    t_end=origin_time+t_after
    
    # Make directory for each unique event waveform
    event_name=str(i+1).zfill(3)
    event_dir=os.path.join(output_dir, event_name)
    os.makedirs(event_dir, exist_ok=True)
    
    # Trim the waveforms
    event_stream=all_streams.copy()
    event_stream.trim(starttime=t_start, endtime=t_end, pad=True, fill_value=0)
    
    for tr in event_stream:
        loc=tr.stats.location if tr.stats.location else ""
        filename=f"{event_name}.{tr.stats.network}.{tr.stats.station}.{loc}.{tr.stats.channel}.SAC"
        filepath=os.path.join(event_dir, filename)
        
        try:
            tr.write(filepath, format="SAC")
        except:
            print(f"Error saving {filename}: {e}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:32<00:00,  1.21it/s]


## Add information to `SAC` header

In [14]:
eventdir_list=sorted(glob("/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/event_waveforms/*"))
for (i, eventdir) in tqdm(enumerate(eventdir_list)):
    sacfile_list=glob(eventdir+"/*.SAC")
    datetime=hypinv_table["   DATE     TIME"][i]
    second=hypinv_table[" SEC "][i]    
    year, month, day=int(datetime[:4]), int(datetime[5:7]), int(datetime[8:10])
    hour, minute, second, microsecond=int(datetime[11:13]), int(datetime[14:16]), int(second), int((second-int(second))*1e6)
    origin_time=utc(year, month, day, hour, minute, second, microsecond)
    evla, evlo=hypinv_table["   LAT  "][i], hypinv_table["   LON   "][i]
    
    for sacfile in sacfile_list:
        code=sacfile.split("/")[-1].split(".")[2]
        stla=used_station_list[used_station_list["Code"]==code].reset_index(drop=True)["Latitude"][0]
        stlo=used_station_list[used_station_list["Code"]==code].reset_index(drop=True)["Longitude"][0]
        tr=read(sacfile)[0]
        tr.stats.sac.evla, tr.stats.sac.evlo=evla, evlo
        tr.stats.sac.stla, tr.stats.sac.stlo=stla ,stlo
        tr.stats.sac.nzyear=origin_time.year
        tr.stats.sac.nzjday=origin_time.julday
        tr.stats.sac.nzhour=origin_time.hour
        tr.stats.sac.nzmin=origin_time.minute
        tr.stats.sac.nzsec=origin_time.second
        
        tr.write(sacfile, format="SAC")

39it [00:03, 10.61it/s]
